## 1 自然语言数据基础

#### 1、这一小节为什么要先学

##### 1.1 在学习 CNN 时，我们先理解图片为什么能输入模型
在学习 CNN 时，我们最开始不是直接学卷积核，而是先学图片本质上是什么、图片为什么能表示成像素矩阵、图片进入模型前为什么要转换成张量。

同样地，在学习 RNN 处理自然语言时，我们也应该先回答几个最基础的问题：

- 自然语言本质上是什么数据
- 计算机能直接理解文字吗
- RNN 为什么不能直接接收一句英文或一句中文
- 文本最终为什么必须变成数字序列

##### 1.2 这一小节的作用
这一小节的作用，就是先把这些最底层的逻辑建立起来。

只有这一步理解清楚了，后面的：

- 分词
- 词表
- 索引化
- Embedding

才不会变成死记硬背。

#### 2、自然语言到底是什么

##### 2.1 对人来说，文本是有意义的语言
例如下面这些句子：

- `I love AI`
- `This movie is great`
- `今天天气很好`

人看到这些句子时，会自然理解：

- 单词是什么意思
- 句子的顺序是什么意思
- 整句话表达的是正面还是负面、陈述还是提问

也就是说，人类看到的是“语义”。

##### 2.2 对计算机来说，文本首先只是符号
但是对于计算机来说，字符串本身并没有天然意义。

例如：

- `love`
- `AI`
- `movie`

这些在计算机内部，最开始都只是字符组成的符号序列，而不是“含义”。

所以这句话非常重要：

人看到的是意义，计算机看到的是符号。

这其实就是自然语言处理的出发点。

##### 2.3 神经网络不能直接计算符号
RNN 本质上仍然是神经网络，而神经网络做的核心操作包括：

- 矩阵乘法
- 向量加法
- 激活函数计算
- 梯度反向传播

这些操作都只能作用在数字上。

所以如果你直接把一句话 `I love AI` 送进去，RNN 是没有办法做计算的，因为字符串不能直接参与矩阵运算。


#### 3、为什么文本不能直接送进 RNN

##### 3.1 RNN 只能接收数值输入
这一点是最核心的。

RNN 的输入不是单词，不是句子，而是“每个时间步上的数值向量”。

也就是说，RNN 真正接收的是：

$x_1, x_2, x_3, \dots, x_T$

其中每个 $x_t$ 都必须是一个向量，例如：

- $x_1 \in \mathbb{R}^{128}$
- $x_2 \in \mathbb{R}^{128}$
- $x_3 \in \mathbb{R}^{128}$

而不是：

- `I`
- `love`
- `AI`

##### 3.2 字符串无法直接参与线性代数运算
例如我们可以计算：

$W \cdot x + b$

但是我们不能计算：

$W \cdot \text{"love"} + b$

因为 `love` 不是数值对象。

所以从模型的角度看，原始文本必须先被转换成数值形式，神经网络才能处理。

##### 3.3 文本长度天然不固定
除了“不是数字”之外，文本还有另一个特点：

句子长度通常不一样。

例如：

- `good` $\rightarrow$ 1 个词
- `I love AI` $\rightarrow$ 3 个词
- `This movie is very interesting` $\rightarrow$ 5 个词

而神经网络在 batch 训练时通常希望输入更规整，所以文本不仅要变成数字，还要进一步变成适合批量处理的统一格式。

这就是为什么后面还会学到：

- 分词
- 建立词表
- 转索引
- Padding
- Embedding


#### 4、自然语言为什么非常适合 RNN

##### 4.1 文本本质上是有顺序的
例如：

`I love AI`

它的顺序是：

`I → love → AI`

这个顺序不能随便打乱。

如果改成：

`AI love I`

虽然单词还是那几个，但句子的语义已经变了，甚至不通顺。

所以文本有一个非常重要的特点：

顺序会影响含义。

而这正好是 RNN 最擅长处理的场景。

##### 4.2 每个词都会受到前文影响
例如句子：

`The movie is not good`

这里最后的 `good` 虽然本身是正向词，但是前面多了一个 `not`，整个句子的情感就变成了负面。

这说明：

- 当前词的理解，常常依赖前面的上下文
- 文本不是孤立特征的堆叠，而是连续的序列

RNN 的设计目标，就是在处理当前时间步时，同时保留前面时间步的信息。

所以它特别适合建模文本这类“前后相关”的数据。

##### 4.3 文本可以看成时间步展开的输入
在 RNN 眼里，一句话其实就像一个时间序列。

例如：

`I love AI`

可以写成：

- 第 1 个时间步输入：`I`
- 第 2 个时间步输入：`love`
- 第 3 个时间步输入：`AI`

所以文本任务中：

- 一个单词或 token 对应一个时间步
- 一句话对应一个输入序列

#### 5、从原始文本到 RNN 输入，中间到底差了什么

##### 5.1 原始文本是人类可读形式
例如：

`I love AI`

这是自然语言原始形式。

它的特点是：

- 便于人理解
- 有语义
- 但不能直接计算

##### 5.2 RNN 需要的是数值向量序列
RNN 实际想要的是这种形式：

$[x_1, x_2, x_3]$

其中：

- $x_1$ 是第一个词的向量
- $x_2$ 是第二个词的向量
- $x_3$ 是第三个词的向量

也就是说，RNN 需要的不是“文本长什么样”，而是“每一步输入对应的数值表示”。

##### 5.3 所以中间一定需要一个转换链条
这个转换链条就是后面我们要逐步学习的内容：

原始文本  
$\rightarrow$ 分词  
$\rightarrow$ 词表映射  
$\rightarrow$ 索引序列  
$\rightarrow$ 向量表示  
$\rightarrow$ 输入 RNN

这一小节先不深入每一步怎么做，但你必须先知道：

文本进 RNN，不是直接进去，而是经过一整套“文本数值化”的过程。

#### 6、把它和 CNN 的输入过程进行类比

##### 6.1 CNN 的输入过程
在 CNN 中：

- 原始图片是 jpg 或 png 文件
- 读取后得到像素值
- 像素值组成矩阵
- 再转成张量
- 最终输入 CNN

也就是说，CNN 也不是直接处理“图片文件”，而是处理“图片对应的数值矩阵”。

##### 6.2 RNN 的输入过程
在 RNN 中：

- 原始文本是句子
- 句子先拆成 token
- token 再转成数字
- 数字再转成向量
- 最终输入 RNN

也就是说，RNN 也不是直接处理“文字本身”，而是处理“文字对应的数值序列”。

##### 6.3 二者本质上是同一个逻辑
- 图片 $\rightarrow$ 像素张量 $\rightarrow$ CNN
- 文本 $\rightarrow$ 向量序列 $\rightarrow$ RNN


#### 7、这一小节最核心的概念：序列

##### 7.1 什么是序列
序列，就是有顺序的一串元素。

例如：

- $[1, 3, 5, 2]$
- `[I, love, AI]`
- 一段语音信号
- 一段时间序列数据

这些都有一个共同特点：

元素的先后顺序是有意义的。

##### 7.2 文本是典型的序列数据
一句话中的词，不是随便摆放的。

例如：

- `dog bites man`
- `man bites dog`

虽然都是这 3 个词，但是顺序不同，意思完全不同。

这说明文本不能像普通表格特征那样，无视顺序地处理。

##### 7.3 RNN 为什么叫 Recurrent
因为它在处理当前输入时，会“带着前面的状态一起往后传”。

也就是说：

- 处理第 2 个词时，会参考第 1 个词的信息
- 处理第 3 个词时，会参考前 2 个词的信息

这正是为了处理“序列”而设计的。

所以文本进入 RNN 的第一前提，就是先把它理解成一个序列，而不是一整个字符串块。